# Video analytics with Intel® Deep Learning Streamer (DL Streamer): Car detection and color classification

Step-by-step **DL Streamer** pipelines on a traffic clip: **car detection** and **color classification**; **OpenVINO™** runs the IR models.


## Pipeline stages

```mermaid
flowchart LR
  IV[Input video] --> Decode[Decode]
  Decode --> Pre[Pre-Process]
  Pre --> Detect[Detection]
  Detect --> Crop[Crop ROIs]
  Crop --> Classify[Classification]
  Classify --> Post[Post-Process]
  Post --> Out[Display (FPS) / Save / Alert]

  style Detect stroke:#d32f2f,stroke-width:3px
  style Classify stroke:#d32f2f,stroke-width:3px
  
```


In [ ]:
from utils import show_config_widgets

show_config_widgets()


### Verify `gvadetect`


In [ ]:
from utils import check_gvadetect
check_gvadetect()


### 1. Display video


This step runs:

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  videoconvert ! \
  autovideosink sync=true
```

In [ ]:
from utils import run_visual, pipeline_raw_video
run_visual(pipeline_raw_video())


### 2. FPS overlay
**`pipeline_fps()`** : adds `gvafpscounter` 

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  videoconvert ! \
  gvafpscounter ! \
  autovideosink sync=true
```


In [ ]:
from utils import run_visual, pipeline_fps
run_visual(pipeline_fps())


### 3. Vehicle detection
**`pipeline_detection()`** : inference step is **`gvadetect`** (car IR), then overlays:

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  gvadetect model=$DETECTION_MODEL device=$DETECTION_DEVICE pre-process-backend=opencv ! \
  gvawatermark ! \
  gvafpscounter ! \
  videoconvert ! \
  autovideosink sync=true
```

In [ ]:
from utils import run_visual, pipeline_detection
run_visual(pipeline_detection())


### 4. Detection + color classification
**`pipeline_detect_classify()`** : **`gvatrack`** + **`gvaclassify`** (color IR) after detection:

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  gvadetect model=$DETECTION_MODEL device=$DETECTION_DEVICE pre-process-backend=opencv ! \
  gvatrack ! \
  gvaclassify model=$CLASSIFICATION_MODEL device=$CLASSIFICATION_DEVICE pre-process-backend=opencv reclassify-interval=$RECLASSIFY_INTERVAL ! \
  queue ! gvawatermark ! gvafpscounter ! \
  videoconvert ! \
  autovideosink sync=true
```

In [ ]:
from utils import run_visual, pipeline_detect_classify
run_visual(pipeline_detect_classify())


### 5. Benchmark: 2 streams


In [ ]:
from utils import run_benchmark, pipeline_benchmark_2
run_benchmark(pipeline_benchmark_2())


### 6. Benchmark: 4 streams


In [ ]:
from utils import run_benchmark, pipeline_benchmark_4
run_benchmark(pipeline_benchmark_4())
